# 90 — EstadoReconcilia lo que hay en Drive contra el manifiesto: qué falta, cuánto ocupa, ysi sobra algo que no debería estar.

## Preámbulo: montar Drive y clonar el repoEl repo es público, así que el clon no necesita credenciales. **Los notebooksllaman a los scripts del repo en vez de reimplementarlos**: el criterio deselección de corridas y el de verificación de ensamblados tienen que vivir enun solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import os, pathlibDRIVE = pathlib.Path('/content/drive/MyDrive/tesis')CLON  = pathlib.Path('/content/tesis')assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'print('Drive OK:', DRIVE)

In [ ]:
import collections, pathlibMAN = DRIVE / '00_manifiestos' / 'srr_manifest.tsv'assert MAN.exists(), f'no hay manifiesto en {MAN} — corré 10_descarga_runs.ipynb'filas = [l.split('\t') for l in MAN.read_text().strip().split('\n')[1:] if l]esperadas = collections.defaultdict(set)for f in filas:    esperadas[f[0]].add(f[1])print(f'manifiesto: {sum(len(v) for v in esperadas.values())} corridas, '      f'{len(esperadas)} organismos')

In [ ]:
SRA = DRIVE / '80_sra'print(f"{'ORG':8} {'TIENE':>6} {'FALTA':>6} {'GB':>8}  SOBRAN")tot_falta = tot_hay = 0for org in sorted(esperadas):    d = SRA / org    hay = {p.stem for p in d.glob('*.sra')} if d.exists() else set()    gb = sum(p.stat().st_size for p in d.glob('*.sra'))/1e9 if d.exists() else 0    falta = esperadas[org] - hay    sobran = hay - esperadas[org]    tot_hay += len(hay & esperadas[org]); tot_falta += len(falta)    print(f'{org:8} {len(hay & esperadas[org]):>6} {len(falta):>6} {gb:>8.1f}  '          f'{sorted(sobran) if sobran else ""}')print(f'\ntotal: {tot_hay} bajadas, {tot_falta} faltan')

Una corrida en la columna **SOBRAN** está en Drive pero no en el manifiesto:o el manifiesto se regeneró con otro criterio, o quedó de una prueba. No laborres sin mirar por qué está.

In [ ]:
for sub in ['70_genomas', '80_sra', '10_bam', '20_yasma']:    d = DRIVE / sub    if not d.exists():        print(f'{sub:12} (no existe)'); continue    n = sum(1 for _ in d.rglob('*') if _.is_file())    gb = sum(p.stat().st_size for p in d.rglob('*') if p.is_file())/1e9    print(f'{sub:12} {n:>5} archivos  {gb:>8.1f} GB')